# Interfacing Python/NumPy with C/C++ and Fortran Code

**Python for HPC course**

2018-2021 Sebastian Ohlmann, Klaus Reuter

Max Planck Computing and Data Facility, Garching

## Motivation

* Call compiled C/C++ or Fortran code from the Python layer
    * Existing (legacy) code
    * Newly developed and optimized code (hotspots, numerical kernels)

$\rightarrow$ combine the convenience of using high-level Python with high-performance compiled code

## Steps

* create software interface between Python/NumPy and C/C++/Fortran
* compile and link to a Python module

## Interfacing Options

* Fortran $\longrightarrow$ **f2py**
* C/C++/CUDA $\longrightarrow$ **Cython**
* C++ $\longrightarrow$ **pybind11**

Some other options (not covered here):

* Swig, an interface generator
* Boost.Python for C++ codes
* Python's low-level C API (different for Python 2 and 3!)

## `f2py`: Interfacing with Fortran code

* `f2py` 
    * is part of NumPy
    * scans the Fortran code and generates signature files (.pyf)
    * automatically takes care of type casting and non-contiguous arrays
* 2 ways of usage
    * direct calling of `f2py`
    * extension in `setup.py` using `numpy.distutils`
* see examples in './f2py' (cf. the [f2py user guide](https://sysbio.ioc.ee/projects/f2py2e/usersguide/index.html))

### Compile and link Fortran code to Python module using `f2py`

* Given the Fortran file `fib.f90`

```Fortran
subroutine fib(a,n)
  integer, intent(in) :: n
  integer(kind=8), intent(out) :: a(n)
  do i=1,n
     if (i.eq.1) then
        a(i) = 0
     elseif (i.eq.2) then
        a(i) = 1
     else 
        a(i) = a(i-1) + a(i-2)
     endif
  enddo
end
```

* Generate '.so' module file by calling `f2py -c fib.f90 -m fib` (second parameter is module name)

### Compile and link Fortran code to Python module using 'setup.py'

* Easy, especially when a `setup.py` already exists
* Necessary change: use `numpy.distutils` instead of `distutils`  
  $\rightarrow$ extension sources may contain a Fortran file

```python
# setup.py
from numpy.distutils.core import setup, Extension

ext = Extension(name = 'fib', sources = ['fib.f90'])

setup(name = 'fibonacci', ext_modules = [ext])
```

```bash
$ python setup.py build_ext --inplace
(...)
$ python -c "import fib; a=fib.fib(16); print(a[-1])"
610
```

## Cython: Interfacing with C/C++ code

### Strategy
* write a Cython extension that defines Python functions which call the external C code
* see the example in the directory `cython/c_interface`

```C
/* foobar/c_hello.h */
void hello(void);
```

```C
/* foobar/c_hello.c */
#include <stdio.h>
#include "c_hello.h"

void hello(void) {
    printf("Hello World!\n");
}
```

```cython
# foobar/hello.pyx
cdef extern from "c_hello.h":
    void hello()

def say_hello():
    hello()
```

```python
# setup.py 
from setuptools import setup, Extension

ext = Extension("foobar.hello",
                sources=["foobar/hello.pyx", "foobar/c_hello.c"])

setup(name="foobar",
      ext_modules=[ext])
```

## Interfacing with C from NumPy-Code using Cython

* Goals
  * Pass NumPy arrays down to C code to perform computation,  
    and get the result back as a NumPy array
  * Leave the memory management to the Python/NumPy layer
* See the comprehensive example in the directory `cython/c_numpy`

### Tipps and tricks for interfacing NumPy with C

* Cython layer
    * check the datatypes to be passed carefully
    * pass the pointer to the NumPy array (`a`) via `a.data` down to the C function
    * handle (temporary) memory allocation conveniently via NumPy arrays here, not in the C layer
    * make sure to only pass contiguous NumPy arrays to C, in case of non-contiguous views create a copy first

* C layer
    * write the extension stateless, beware of memory leaks
    * you may use all kinds of code optimization techniques
        * OpenMP threading (not affected by the Python GIL)
        * vectorization
        * cache optimization, etc.
    * tweak the compiler flags in `setup.py`

## Interfacing with a C library (shared object) using Cython
* Goal: Use a C library from Python
    * third-party library for which no Python bindings exist, yet
    * legacy code you want to wrap into Python
    * CUDA code (compile into `.so` file independently using `nvcc`, first)
* see the comprehensive example in `cython/c_interface_shared_object`

### Tipps and tricks for interfacing shared objects
* `setup.py`
    * usability: locate the library and its headers, options
        * installation location passed via environment variable,  
          often the case on HPC systems via environment modules (`MKL_ROOT`, `GSL_HOME`, `FFTW_HOME`)
        * installation location passed as arguments to `setup.py`
        * installed at sytem location
    * deployment: add the RPATH to the link line pointing to library location, to avoid `LD_LIBRARY_PATH`
* for CUDA code, it is much easier to go via a library than using `setup.py` with `nvcc`

## Cython and C++

* Cython supports most of the C++ features: classes, templates, overloading
* typical use case: write a Python wrapper class for a C++ class
* not covered here, please see for further details  
  http://cython.readthedocs.io/en/latest/src/userguide/wrapping_CPlusPlus.html

### Cython provides interfaces to libc and c++ STL

In [1]:
%load_ext Cython

In [2]:
%%cython
from libc.math cimport sin

print(sin(1.0))

0.8414709848078965


In [3]:
%%cython
# distutils: language = c++
from libcpp.vector cimport vector

cdef vector[int] v = range(10)
print(v)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## pybind11